# Phase 1 – Annotationsstudie
Die Annotationsstudie prüft anhand einer festgelegten Stichprobe, in welchem Umfang die auf Volltextebene annotierten SemEval-Policy-Frames bereits auf Ebende der Nachrichtentitel erkennbar sind. Hierzu werden menschliche und LLM-basierte Titelannotationen miteinander sowie mit den SemEval-Volltextannotationen verglichen.

## Notebook A - Datenaufbereitung, Übersetzung und Sampling
### Verortung im Gesamtprojekt
Dieses Notebook führt die vorbereitenden Schritte der Annotationsstudie aus:

1. Es durchsucht die konfigurierten SemEval-Sprachordner und die Train-/Dev-Unterordner.
2. Es liest je Artikel die erste nichtleere Zeile als Nachrichtentitel aus.
3. Es übersetzt die Titel mit dem Google-Translator ins Deutsche.
4. Es erzeugt eine Excel-Datei mit Artikeldatei, Originaltitel, deutschem Titel, Sprachkennung und den 14 vorgesehenen Frame-Spalten.
5. Es zieht innerhalb jeder Sprache ein 15-%-Sample mit `random_state=42` und speichert dieses als eigene Excel-Datei.

## Wichtige Abgrenzung

Die hier erzeugte Stichprobe gehört ausschließlich zur Annotationsstudie. Zusätzliche Daten, die später für das Modelltraining aufbereitet werden, dürfen die Phase-1-Stichprobe und ihre Ergebnisse nicht verändern.

In [ ]:
# !uv pip install openpyxl deep-translator tqdm

In [ ]:
import os
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm

In [ ]:
# ---- config
BASE_DIR = '..\0_data'

STATIC_COLUMNS = [
    "Economic",
    "Capacity and resources",
    "Morality",
    "Fairness and equality",
    "Legality, constitutionality and jurisprudence",
    "Policy prescription and evaluation",
    "Crime and punishment",
    "Security and defense",
    "Health and safety",
    "Quality of life",
    "Cultural identity",
    "Public opinion",
    "Political",
    "External regulation and reputation"
]

TARGET_SUBFOLDERS = ['dev-articles-subtask-2', 'train-articles-subtask-2']

<positron-console-cell-3>:2: SyntaxWarning: invalid escape sequence '\d'


In [ ]:
# ---- gather all file paths first so tqdm knows the exact total
files_to_process = []
language_folders = ['en', 'fr', 'ge', 'it', 'po']

for lang in language_folders:

    lang_path = os.path.join(BASE_DIR, lang)

    for subfolder in TARGET_SUBFOLDERS:

        subfolder_path = os.path.join(lang_path, subfolder)

        if os.path.isdir(subfolder_path):
            for filename in os.listdir(subfolder_path):
                if filename.endswith('.txt'):
                    files_to_process.append({
                        'filepath': os.path.join(subfolder_path, filename),
                        'language': lang,
                        'filename': filename
                    })

print(f"Found {len(files_to_process)} articles to process across {len(language_folders)} languages.")

Found 1401 articles to process across 5 languages.


In [ ]:
# ---- prepare translation loop
data = []
translator = GoogleTranslator(source='auto', target='de')

# ---- full loop: wrap the list in tqdm() for progress control
for file_info in tqdm(files_to_process, desc="Translating Articles", unit="file"):

    # read the file
    with open(file_info['filepath'], 'r', encoding='utf-8') as file:
        lines = file.readlines()

    # extract headline
    headline = ""
    for line in lines:
        clean_line = line.strip() # remove trailing lines
        if clean_line:
            headline = clean_line
            break

    # translate
    translated_headline = translator.translate(headline)

    # build row
    row = {
        "Filename": file_info['filename'],
        "Headline": headline,
        "Headline_GER": translated_headline,
        "Language": file_info['language']
    }

    # add static columns
    for col in STATIC_COLUMNS:
        row[col] = ""

    data.append(row)

Translating Articles: 100%|██████████| 1401/1401 [17:02<00:00,  1.37file/s]


In [ ]:
# ---- generate excel file
df = pd.DataFrame(data)
ordered_columns = ["Filename", "Headline", "Headline_GER", "Language"] + STATIC_COLUMNS
df = df[ordered_columns]

output_filename = 'headlines_dataset.xlsx'
df.to_excel(output_filename, index=False)

In [ ]:
# ---- load the original excel file
input_filename = 'headlines_dataset.xlsx'
df = pd.read_excel(input_filename)

# ---- draw a random 15% sample per language group
sampled_df = df.groupby('Language', group_keys=False).sample(frac=0.15, random_state=42)

In [ ]:
# ---- print verification of the sampling counts
print('Original counts per language:')
print(df['Language'].value_counts())
print('\n15% Sample counts per language:')
print(sampled_df['Language'].value_counts())

# ---- save the reduced sample to a new excel file
output_filename = 'headlines_dataset_15sample.xlsx'
sampled_df.to_excel(output_filename, index=False)

Original counts per language:
Language
en    516
it    303
fr    211
po    194
ge    177
Name: count, dtype: int64

15% Sample counts per language:
Language
en    77
it    45
fr    32
po    29
ge    27
Name: count, dtype: int64
